What changes from P1 to P2

- Nodes per element:   3             | 6
- Basis functions:     linear        | quadratic
- Gradient:            constant      | linear
- Local matrices:      explformula   | quadrature
- Reference element:   optional      | essential
- Jacobian:            rarely needed | required
- Quadrature:          not necessary | necessary

**Stays the same**:
- mesh generation
- global assembly
- Dirichlet boundary condition
- generalised eigenproblem
- convergence study
- visualisation

**Changes**:
- local basis functions
- local stiffness matrix
- local mass matrix
- element geometry representation
- numerical integration


In [14]:
# 2D let the domain be uniform and defined on [0, 1]x[0, 1]
import numpy as np
import sympy as sp
from scipy.linalg import eigh
from scipy.spatial import Delaunay
import pandas as pd
import math
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [15]:
### FEM METHODS ###

def generate_mesh(n_points: int):

       '''
       Generates a 2D mesh over the unit square [0, 1] x [0, 1]

       Places n_points evenly along each axis, builds coordinate grid, 
              and divides into triangulations using Delaunay triangulation
       
       :param n_points: a number of points along each axis
       :return: returns domain - an (N, 2) array of 2D domain coordinates 
              and tri - nodes of the trinagle in the global domain
       '''

       # define the axis intervals
       x = np.linspace(0, 1, n_points)
       y = np.linspace(0, 1, n_points)

       # coordinate generation via meshgrid (takes 1D arrays and duplicates them to build 2D grids)
       X, Y = np.meshgrid(x, y)

       # c_ matches the first X with the first Y, second X with second Y etc
       # ravel() takes 2D matrix structure and reads it row by row into a long single list of coordinates
       # Delaunay cannot read 2D grid, thats why we flatten X and Y, so they can be paired together
       domain = np.c_[X.ravel(), Y.ravel()]
       

       # create triangles on the domain
       tri = Delaunay(domain)

       return domain, tri

def add_mid_pts(domain, triangles_nodes):

       
       mid_pts = {}

       for triangle in triangles_nodes:
              a, b, c = triangle

              edges = [
                     tuple(sorted((a, b))),
                     tuple(sorted((b, c))),
                     tuple(sorted((c, a)))
              ]
       
              for edge in edges:
                     if edge not in mid_pts:
                            i, j = edge

                            midpoint = (domain[i] + domain[j]) / 2
                            mid_pts[edge] = midpoint


       return mid_pts

def get_p2_triangles(domain, triangles_nodes, mid_pts):
       
       edge_to_midpoint = {}
       i = 0
       for nodes in mid_pts.keys():
              edge = nodes
              edge_to_midpoint[edge] = len(domain) + i
              i += 1
       
       p2_triangles_nodes = []

       for triangle in triangles_nodes:
              p2_nodes = []

              a, b, c = triangle
              d = edge_to_midpoint[tuple(sorted((a, b)))]
              e = edge_to_midpoint[tuple(sorted((b, c)))]
              f = edge_to_midpoint[tuple(sorted((c, a)))]

              p2_nodes.append(a)
              p2_nodes.append(b)
              p2_nodes.append(c)
              p2_nodes.append(d)
              p2_nodes.append(e)
              p2_nodes.append(f)

              p2_triangles_nodes.append(p2_nodes)
       
       p2_triangles_coords = []

       for triangle in triangles_nodes:
              p2_coords = []
              a, b, c = triangle
              p2_coords.append(domain[a])
              p2_coords.append(domain[b])
              p2_coords.append(domain[c])

              p2_coords.append(mid_pts[tuple(sorted((a, b)))])
              p2_coords.append(mid_pts[tuple(sorted((b, c)))])
              p2_coords.append(mid_pts[tuple(sorted((c, a)))])

              p2_triangles_coords.append(p2_coords)


       return np.array(p2_triangles_nodes), np.array(p2_triangles_coords)


       

def stiffness_matrix_A(grad_phi):

       pass

def create_symmetric_matrix(lower_tri_matrix):

       '''
       This method creates a symmetric matrix out of lower triangulated matrix 
              by adding it with its transpose
       
       :param lower_tri_matrix: a lower triangular matrix
       :return: returns a symmetric matrix
       '''

       # Zero the diagonal so it isnt double counted, when added
       lower_tri_no_diag = lower_tri_matrix.copy()
       np.fill_diagonal(lower_tri_no_diag, 0)
       
       # create symmetric matrix by adding the lower triangular matrix to its transpose
       sym_matrix = lower_tri_no_diag + lower_tri_no_diag.T

       return sym_matrix


def mass_matrix_M():

       ''' 
       mass_matrix_M stores the local mass matrix, because it is the same for every mesh, 
              the only difference would be the scalling by the area of the triangle, 
              which is handled by triangle_solver
       
       :return: returns a local mass matrix
       '''
       
       pass



def local_triangle_matrices(coords_of_triangle):

       '''
       local_triangle_matrices receives the coordinates of the triangle in the domain and calculates:
              the stiffness matrix - measures the allignment of the gradients of each basis function on this triangle
              the mass matrix - measures the overlap of the heights of each basis function on this triangle
       
       :param coords_of_triangle: the (N, 2) numpy array of the global coordintes of the triangle
       :return: returns the computed local stiffness and mass matrices
       '''

       # find the area of the triangle
       col = np.array([1, 1, 1])
       # create 3x3 matrix to find the area
       coords_matrix = np.hstack((coords_of_triangle, np.atleast_2d(col).T))

       # area of a triangle
       T_k = 0.5 * abs(np.linalg.det(coords_matrix))

       x = []
       y = []
       # for coordinate in all of the coordinates of the nodes of this triangle
       for coord in coords_of_triangle:
              x.append(float(coord[0]))
              y.append(float(coord[1]))

       # basis function has an equation phi = a + bx + cy, where b and c are the coefficients of the coordinates
       # so we find b and c for every basis function of the triangle
       c = []   
       c.append(x[2] - x[1])
       c.append(x[0] - x[2])
       c.append(x[1] - x[0])

       b = []
       b.append(y[1] - y[2])
       b.append(y[2] - y[0])
       b.append(y[0] - y[1])

       # find the gradients of the basis functions
       grad_phi = np.array([
              (1 / (2 * T_k)) * np.array([b_i, c_i]) for b_i, c_i in zip(b, c)
       ])


       # get local stiffness matrix
       A_local = T_k * stiffness_matrix_A(grad_phi)
       
       # get local mass matrix
       M_local = (T_k / 12) * mass_matrix_M()
       
       return A_local, M_local

# put the triangle in the global matrix
def put_local_to_global(global_matrix, local_matrix, coord):

       '''
       This method puts local matrices into the corresponding global ones according to their position in the domain

       :param global_matrix: the global matrix (the matrix of the domain) either empty if the first node or already have previous local matrices in it
       :param local_matrix: the matrix of the triangle
       :param coord: the coordinates of the triangle in the domain

       :return: return the obtained global matrix
       '''

       n_local = local_matrix.shape[0]

       # we need to put every value of the local matrix to the global, that's why we need 2 loops: one for rows, another for columns
       for a in range(n_local):
              for b in range(n_local):
                     global_matrix[coord[a], coord[b]] += local_matrix[a, b]
       

       return global_matrix

# get global matrices
def get_global_matrices(tri_coord_sort, domain, n_nodes):

       '''
       This method obtains the global matrices by putting local matrices in the global

       :param tri_coord_sort: a list of sorted global nodes of the triangle
       :param domain: the list of all coordinates of the nodes
       :param n_nodes: the number of nodes that will be used as the rank for the global matrix
       
       :return: returns obtained global stiffness and mass matrices
       '''

       A_global = np.zeros((n_nodes, n_nodes), dtype=float)
       M_global = np.zeros((n_nodes, n_nodes), dtype=float)

       # for every triangle in the mesh
       for triangle in tri_coord_sort:
              coords = domain[triangle]

              A_local, M_local = local_triangle_matrices(coords)
              global_coords = triangle.tolist()

              put_local_to_global(A_global, A_local, global_coords)
              put_local_to_global(M_global, M_local, global_coords)

       return A_global, M_global

def get_boundary_and_interior_nodes(domain):

       '''
       This method finds the nodes that are on the boundary and that are inside of the domain

       :param domain: the (N, 2) array of all coordinates of the nodes in this domain
       :return: returns a list of the nodes that are on the boundary and list of the interior nodes
       '''

       boundary_nodes = []
       interior_nodes = []

       # get the node and its coordinates
       for i, (x, y) in enumerate(domain):
              # check if any of the coordinates are on the boundary
              if x == 0 or x == 1 or y == 0 or y == 1:
                     boundary_nodes.append(i)
              else:
                     interior_nodes.append(i)

       return boundary_nodes, interior_nodes


def apply_dirichlet(A_global, M_global, interior_nodes):

       '''
       apply_dirichlet is the method that reduces the global matrices according to the Dirichet Boundary Condition
              all the nodes on the boundary will be excluded from the global matrices because they will be zero

       :param A_global: global stiffness matrix
       :param M_global: global mass matrix
       :param interior_nodes: the list of the interior nodes

       :return: returns the reduced global stiffness and mass martices based on the interior nodes of the domain
       '''

       # reducing matrices based on boundary condition, that u = 0 on the boundary
       A_reduced = A_global[np.ix_(interior_nodes, interior_nodes)]
       M_reduced = M_global[np.ix_(interior_nodes, interior_nodes)]

       return A_reduced, M_reduced



In [16]:
### NUMERICAL ERRORS ###

def first_eigval_error(comp_eigval):

       '''
       This method calculates the error of the first eigenvalue

       :param comp_eigval: the first eigenvalue that was obtained numerically
       :return: returns the difference between the exact and numerical values
       '''

       real_eigval = 2 * (np.pi)**2
       error = abs(real_eigval - comp_eigval)
       
       return error

def second_third_eigval_error(comp_eigval_second, comp_eigval_third):

       '''
       This method calculates the error of the second and third eigenvalues
              because for the unit square the second and the third values are repetititve, 
                     therefore we need to compare both of them to the exact value

       :param comp_eigval_second: the second eigenvalue that was obtained numerically
       :param comp_eigval_third: the third eigenvalue that was obtained numerically

       :return: returns the difference between the exact and numerical values
       '''

       real_eigval = 5 * (np.pi)**2
       error_second = abs(real_eigval - comp_eigval_second)
       error_third = abs(real_eigval - comp_eigval_third)
       # taking the average between second and third because comparing the same real eigenvalue
       error = (error_second + error_third) / 2

       return error



In [17]:
### VISUALISATIONS ###

def visualise_mesh(domain, triangle):

       '''
       visualise_mesh will help to see the location of the nodes and local triangles

       :param domain: the list of coordinates of all nodes
       :param triangle: the list of coordinates of the nodes of the local triangle
       '''

       plt.triplot(domain[:,0], domain[:,1], triangle.simplices.copy())
       plt.plot(domain[:,0], domain[:,1], "o")

       # to see the nodes on the graph
       # enumerate shows which node number corresponds to the coordinates
       for i, (x, y) in enumerate(domain):
              plt.text(x, y, f"P{i}", fontsize=9)

       # labeling the triangles
       for k, tri in enumerate(triangle.simplices):

              centroid = domain[tri].mean(axis=0)

              plt.text(centroid[0], centroid[1], f"T{k}", color="red", fontsize=6)

       # gca - get current axes
       plt.gca().set_title("Mesh visualisation")
       
       # set_aspect("equal") prevents stretching the plot, if it's square it will look like square
       plt.gca().set_aspect("equal")
       plt.show()

# visualise the FEM function reconstructed from the nodal values
def visualise_FEM(eigenvectors, domain, tri, n_nodes, interior_nodes):

       '''
       This method visualises the numerical functions that were found using finite element method

       :param eigenvectors: the coordinates of the eigenvector that were obtained numerically
       :param domain: the list of coordinates of all nodes
       :param tri: triangulations
       :param n_nodes: number of the nodes
       :param interior_nodes: the list of the interior nodes (that are not on the boundary)
       '''

       # after the 3rd the graphs become unnecessary because of the lack of precision
       for k in range(min(3, eigenvectors.shape[1])):
              # get k-th eigenvector of the global reduced system
              v_k = eigenvectors[:, k]

              u = np.zeros(n_nodes)
              u[interior_nodes] = v_k # use interior nodes, because boundary nodes will vanish due to Dirichet boundary condition


              plt.tripcolor(
                     domain[:, 0], # x axis
                     domain[:, 1], # y axis
                     tri.simplices, # will give the coordinates of the triangles
                     u,
                     shading="gouraud" # will give the smooth graph, where colours will change gradually
                     # can also use flat
              )

              # will help to see the difference in heights of the eigenfunctions
              plt.colorbar()
              plt.gca().set_aspect("equal")
              plt.gca().set_title(f"Eigenfunction with {k+1}-th eigenvector")
              
              plt.show()

              

def visualise_convergence(dofs, errors):

       '''
       This method plots the convergence of the error against the degrees of freedom.
              In Finite Element method the number of degrees of freedom (number of the interior nodes in the mesh)
              determines the accuracy of the approximation - a finer mesh means more nodes, 
              which reduces the error
              
       :param dofs: a list of degrees of freedom (interior nodes)
       :param errors: a list of errors
       '''

       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(dofs, errors, marker="o")
       ax.set_xlabel(r"Degrees of Freedom")
       ax.set_ylabel(r"$E(h) = |\lambda_1 - \lambda_1^h|$") # using LaTeX to express the equation for finding the error
       ax.set_title("Convergence of the FEM eigenvalues")
       ax.grid(True)
       plt.show()

def visualise_convergence_rate(base, h, errors):

       '''
       This method shows the graph of the convergence rate

       :param base: the base of the logarithm which was calculated by the ratios between the neighbouring results
       :param h: a list of the steps between the nodes, calculated as h = 1 / (n_nodes - 1)
       :param errors: a list of errors 
       '''

       fig, ax = plt.subplots(figsize=(10, 8))
       # using the log with a special base because ratio between number of points may differ
       ax.plot(np.emath.logn(base, h), np.emath.logn(base, errors), marker="o")
       ax.set_xlabel(f"log_{base:.3f}(h)")
       ax.set_ylabel(f"log_{base:.3f} (E(h))")
       ax.set_title("Convergence rate of the FEM eigenvalues")
       ax.grid(True)
       plt.show()

def visualise_eigenfunctions(discrete_eigf, exact_eigf, interior_nodes):

       '''
       Shows the comparison between numerically obtained eigenfunction and exact

       :param discrete_eigf: the eigenfunction that was computed numerically
       :param exact_eigf: the exact eigenfunction
       :param interior_nodes: the list of the interior nodes
       '''
       
       # reshape eigenvectors into square arrays

       # boundary nodes will vanish at the boundary, so we need to know the length of the side of the square produced by interior nodes
       interior_dim = int(np.sqrt(len(interior_nodes))) # int to prevent it being a float, because it won't be accepted when the eigenfunction matrix will be reshaped
       discrete_eigf = discrete_eigf.reshape(interior_dim, interior_dim)
       exact_eigf = exact_eigf.reshape(interior_dim, interior_dim)
       
       fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
       fig.suptitle("Numerical Eigenfunction & Exact Eigenfunction")
       
       ax1.imshow(discrete_eigf)
       ax1.set_title("FEM")

       ax2.imshow(exact_eigf)
       ax2.set_title("Exact")

       plt.show()
       

In [18]:
### CHECKS FOR MATRIX SYMMETRY AND ROW SUM CHECK ###
# for finite element method the stiffness and mass matrices should be symmetric
# and row sum should be equal to 0

def sanity_check(A_global, M_global):
       
       # check if the matrices are symmetric and their row sum is 0 (the sum might have some floating point error that's why we round it)
       if np.allclose(A_global, A_global.T) == True and np.allclose(M_global, M_global.T) == True and np.allclose(A_global.sum(axis=1), 0) == True:
              return True
       else:
              return False


In [19]:
### CONVERGENCE STUDY ###

# find local convergence rate
def local_convergence_rate(row_num, n_points_list, errors):
       
       '''
       This method finds the local convergence rate (between two neighbouring rates), 
       which in other words means "how fast are we approaching the exact value"

       :param row_num: detects where are we in the table of convergence
       :param n_points_list: list of numbers of points, 
       we will need them to find the ratio between the number of points because it will determine the convergence rate
       :param errors: a list of errors that we got from comparing the exact value with numerical

       :return: returns the local rate of convergence or
       None if it is the first row or if the error is 0 or negative to ensure that we won't face division by nothing or zero
       '''

       if row_num == 0:
              
              return np.nan
              
       # prevents the cases when the error is 0
       e_old = errors[row_num-1]
       e_new = errors[row_num]
       if e_old <= 0 or e_new <= 0:
              return np.nan

       # find the local convergence rate

       # we need to find the base for the logarithm first
       ratio_between_num_points = n_points_list[row_num] / n_points_list[row_num-1] 
       # find the ratio between 2 neighbouring local errors
       ratio_between_errors = errors[row_num-1] / errors[row_num]
       # calculate the local rate of convergence
       p_local = math.log(ratio_between_errors, ratio_between_num_points)

       return p_local



In [20]:
### COMPARE NUMERICAL AND EXACT EIGENFUNCTIONS ###

# we need to evaluate it at every node and compare with first eigenvector
def compare_numerical_with_exact(node_coords, u_h):

       '''
       This method compares the first eigenfunction with the first exact eigenfunction which for the unit square is
       u(x,y) = sin(pi * x) * sin(pi * y)

       :param node_coords: the (N, 2) array of the (x, y) coordinates of the nodes of the domain
       :param u_h: the numerically computed eigenfunction

       :return: returns 2 values: first is the list of exact values of the function at each node
       second is the numerical eigenfunction
       '''
       
       u_exact = []

       for node_coord in node_coords:
              x = node_coord[0]
              y = node_coord[1]
              
              # calculate the exact eigenfunction
              u_exact.append(np.sin(np.pi * x) * np.sin(np.pi * y))

              
       u_exact = np.array(u_exact)

       # handle the sign (because eigenfucntions are only determined up to a sign), 
       # so two vectors that represent the same function pointing in opposite direction (dot product is < 0) this will give a huge error when calculating the norm
       if np.dot(u_exact, u_h) < 0:
              u_h = -u_h

       # normalise to work in the same ratios
       u_exact /= np.linalg.norm(u_exact)
       u_h /= np.linalg.norm(u_h)
       
       return u_exact, u_h

def compare_second_third_eigf_with_exact(node_coords, u_h_2, u_h_3):

       '''
       This method compares the second and third eigenfunction with the second and third exact eigenfunction which for the unit square is
       u(x,y) = sin(2 * pi * x) * sin(pi * y) or u(x,y) = sin(pi * x) * sin(2 * pi * y)

       :param node_coords: the (N, 2) array of the (x, y) coordinates of the nodes of the domain
       :param u_h_2: the numerically computed second eigenfunction
       :param u_h_3: the numerically computed third eigenfunction

       :return: returns 3 values: first is the list of exact values of the function at each node
       second and third are the numerical eigenfunctions
       
       '''

       exact_12 = []
       exact_21 = []

       for node_coord in node_coords:
              x = node_coord[0]
              y = node_coord[1]
              
              # calculate the exact eigenfunction
              exact_12.append(np.sin(np.pi * x) * np.sin(2 * np.pi * y))
       
              exact_21.append(np.sin(2 * np.pi * x) * np.sin(np.pi * y))

       exact_12 = np.array(exact_12)
       exact_21 = np.array(exact_21)

       # handle the sign (because eigenfucntions are only determined up to a sign), 
       # so two vectors that represent the same function pointing in opposite direction (dot product is < 0) this will give a huge error when calculating the norm
       if np.dot(exact_12, u_h_2) < 0:
              u_h_2 = -u_h_2
       if np.dot(exact_21, u_h_3) < 0:
              u_h_3 = -u_h_3

       # normalise to work in the same ratios
       exact_12 /= np.linalg.norm(exact_12)
       exact_21 /= np.linalg.norm(exact_21)
       u_h_2 /= np.linalg.norm(u_h_2)
       u_h_3 /= np.linalg.norm(u_h_3)

       return exact_12, exact_21, u_h_2, u_h_3

In [21]:
### THE FULL FEM ANALYSIS ###

def run_FEM_P2_analysis(n_points, n_points_list, errors, errors_second_third, errors_eigf, index):

       '''
       This method combines all the methods to perform finite element method computations for the unit square

       :param n_points: the number of points per axis in the unit square
       :param n_points_list: the list of number of points per axis in the unit square
       :param errors: the list of errors between the exact and numerical values
       :param errors_second_third: the list of errors between the exact eigenvalue and the second and third
       (the exact eigenvalue would repeat itself, that's why we compare with second and third)
       :param errors_eigf: the list of errors between numerical eigenfunctions and exact
       :param index: shows what round of the loop we are having (will need for tables)

       :return: returns a dictionary that stores:
       h - the step between the nodes
       n_points - the number of points per axis
       dofs - list of degrees of freedom (the number of the interior nodes)
       eigval - the first eigenvalue
       error_eigval - the error of the first eigenvalue
       error_23 - the error of the second and third eigenvalues against the exact eigenvalue
       error_eigf - the error of the approximation of the exact eigenfunction
       p_eigval - the convergence rate of the eigenvalues (how fast they approach the exact value)
       p_eigf - the convergence rate of the eigenfunctions (how fast do the approximations approach the function)
       second_eigvec_second_eigf - the dot product of the second eigenvector and second exact eigenfunction
       second_eigvec_third_eigf - the dot product of the second eigenvector and third exact eigenfunction
       third_eigvec_second_eigf - the dot product of the third eigenvector and second exact eigenfunction
       third_eigvec_third_eigf - the dot product of the third eigenvector and third exact eigenfunction
       '''
       
       ### Mesh Generation

       # generate mesh and split it into triangles
       domain, triangle = generate_mesh(n_points)
       # sort nodes of each triangle into ascending order so they are consistently ordered
       tri_coord_sort = np.sort(triangle.simplices)

       print(tri_coord_sort)

       # optional visualisation for small meshes
       if n_points < 15: # everything above 15 becomes indistinguishable on the plot
              visualise_mesh(domain, triangle)


       ### Getting Global Matrices

       n_nodes = len(domain)
       A_global, M_global = get_global_matrices(tri_coord_sort, domain, n_nodes)
       
       # check if the matrices are symmetric and if the row sum is 0
       if sanity_check(A_global, M_global) == False:
              print("The global matrices are not symmetric or their row sum is not zero")
              print("There is an error")
              raise ValueError

       

       ### Boundary conditions

       boundary_nodes, interior_nodes = get_boundary_and_interior_nodes(domain)
       # apply dirichlet BC
       A_reduced, M_reduced = apply_dirichlet(A_global, M_global, interior_nodes)

       
       ### Eigenvalues

       # finding eigenvalues and its errors
       # SciPy stores eigvectors as columns
       eigvals, eigvecs = eigh(A_reduced, M_reduced)

       print(f"The eigenvalues: \n{eigvals[0]}\n")

       # eigenvalue error
       error = first_eigval_error(eigvals[0])
       errors.append(error)
       if n_points < 4: # prevents the error, because with 3 points per axis there is only one eigenvalue
              errors_second_third.append(0)
       else:
              error_23 = second_third_eigval_error(eigvals[1], eigvals[2])
              errors_second_third.append(error_23)

       
       ### Eigenfunctions

       # eigenfunction comparison
       interior_nodes_coords = [domain[interior_node] for interior_node in interior_nodes] 
       exact_eigf, discrete_eigf = compare_numerical_with_exact(interior_nodes_coords, eigvecs[:,0])
       
       # compare second and third with the exact
       if n_points < 4: # for 3 points we won't get the second and third eigenvectors
              exact_12, exact_21, v2, v3 = 0, 0, 0, 0
       else:
              exact_12, exact_21, v2, v3 = compare_second_third_eigf_with_exact(interior_nodes_coords, eigvecs[:,1], eigvecs[:, 2])

       # get the L2 error of the exact eigenfunction and numerical
       error_eigf = np.linalg.norm(exact_eigf - discrete_eigf)
       errors_eigf.append(error_eigf)

       
       ### Convergence

       h = 1 / (n_points - 1)
       p_eigval = local_convergence_rate(index, n_points_list, errors)
       p_eigf = local_convergence_rate(index, n_points_list, errors_eigf)

       
       ### Visualisations

       # visualise FEM
       visualise_FEM(eigvecs, domain, triangle, n_nodes, interior_nodes)
       # visualise exact eigenfunction next to FEM eigenfunction
       visualise_eigenfunctions(discrete_eigf, exact_eigf, interior_nodes)
       

       return {
              "h": h,
              "n_points": n_points,
              "dofs": len(interior_nodes),
              "eigval": eigvals[0],
              "error_eigval": error,
              "error_23": errors_second_third,
              "error_eigf": error_eigf,
              "p_eigval": p_eigval,
              "p_eigf": p_eigf,
              "second_eigvec_second_eigf": abs(np.dot(v2, exact_12)),
              "second_eigvec_third_eigf": abs(np.dot(v2, exact_21)),
              "third_eigvec_second_eigf": abs(np.dot(v3, exact_12)),
              "third_eigvec_third_eigf": abs(np.dot(v3, exact_21))

       }

In [22]:
### DERIVATION OF THE P2 LAGRANGE BASIS FUNCTIONS ON THE REF TRIANGLE ###

# we have to find the basis functions from the reference coordinates
# every quadratic polynomial on the ref triangle can be expressed as
# u(xi, eta) = c1 + c2*xi + c3*eta + c4*xi^2 + c5*xi*eta + c6*eta^2
#
# we evaluate at every point of the reference triangle with 3 vertices and 3 midpoints of the edges
# this gives us a system of parametric equations 
# U = A * c, 
# where U = [u1, ..., u6]^T - nodal values, 
# A - Vandermonde matrix
# c = [c1, ..., c6]^T - polynomial coefficients
#  
# solving for c will give us the polynomial coefficients in terms of nodal values
# 
# substituting back to u(xi, eta) will give us
#
# u_h = sum_i phi_(xi, eta) * u_i
#
# where phi_i are the P2 basis fucntions

### IMPORTANT
# u(xi, eta) = general quadratic polynomial
# u_h(xi, eta) = finite element interpolant
# phi_i(xi, eta) = basis functions


def u(xi, eta):
       return (c1 + c2*xi + c3*eta + c4*xi**2 + c5*xi*eta + c6*eta**2)


# will treat them as variables and wont raise an udefined error
c1, c2, c3, c4, c5, c6 = sp.symbols("c1 c2 c3 c4 c5 c6")
u1, u2, u3, u4, u5, u6 = sp.symbols("u1 u2 u3 u4 u5 u6")
xi, eta = sp.symbols("xi eta")

def derive_p2_basis_func(show_matrices=False):
       # define ref coords
       ref_pts_coords = [
              (0, 0),
              (1, 0),
              (0, 1),
              (sp.Rational(1, 2), 0),
              (sp.Rational(1, 2), sp.Rational(1, 2)),
              (0, sp.Rational(1, 2))
       ]


       A, U = get_Vand_and_U_matrices(ref_pts_coords)
       if show_matrices == True:
              sp.pprint(U)
              sp.pprint(A)

       # now going other way, to find c1, ..., c2 using u1, ..., u6
       u_h = get_u_h(A)

       # now returning back to find u1, ..., u6 that are expressed in terms ofxi, eta and c1, ..., c6 that are expressed in terms of u1, ..., u6
       basis_functions = [
              sp.simplify(u_h.coeff(ui))
              for ui in [u1, u2, u3, u4, u5, u6]
       ]

       # basis functions matrix

       return sp.Matrix(basis_functions)     



def get_Vand_and_U_matrices(ref_pts_coords):
       # matrix obtained with unknown coef c1, ..., c6
       U = sp.Matrix([
              u(xi, eta)
              for xi, eta in ref_pts_coords
       ])

       # Vandermonde matrix 
       A = sp.Matrix([
              [1, xi, eta, xi**2, xi*eta, eta**2]
              for xi, eta in ref_pts_coords
       ])
       
       return A, U



# approxiate u using the coefficients c1, ..., c6
def get_u_h(A):
       
       # define unknown nodal fucntions
       U_nodal = sp.Matrix([
              u1,
              u2,
              u3,
              u4,
              u5,
              u6
       ])

       A_inv = A.inv()
       c = A_inv * U_nodal

       # approximate the basis function in terms of unknown c1, ..., c6 and ref coords xi and eta
       u_h = (
              c[0] + c[1]*xi + c[2]*eta + c[3]*xi**2 + c[4]*xi*eta + c[5]*eta**2
       )

       return sp.expand(u_h)


In [23]:
### GRADIENTS OF THE BASIS FUNCTIONS ###

def basis_func_grad(B_phi):
       xi, eta = sp.symbols("xi eta")

       # for every basis function put the partial derivatives in this matrix
       B_grad_phi = sp.Matrix([
              [sp.diff(phi, xi), sp.diff(phi, eta)]
              for phi in B_phi
       ])
       
       
       return B_grad_phi
       
B_phi = derive_p2_basis_func()
B_grad_phi = basis_func_grad(B_phi)

sp.pprint(B_phi)
sp.pprint(B_grad_phi)

⎡   2                    2          ⎤
⎢2⋅η  + 4⋅η⋅ξ - 3⋅η + 2⋅ξ  - 3⋅ξ + 1⎥
⎢                                   ⎥
⎢            ξ⋅(2⋅ξ - 1)            ⎥
⎢                                   ⎥
⎢            η⋅(2⋅η - 1)            ⎥
⎢                                   ⎥
⎢         4⋅ξ⋅(-η - ξ + 1)          ⎥
⎢                                   ⎥
⎢               4⋅η⋅ξ               ⎥
⎢                                   ⎥
⎣         4⋅η⋅(-η - ξ + 1)          ⎦
⎡4⋅η + 4⋅ξ - 3   4⋅η + 4⋅ξ - 3 ⎤
⎢                              ⎥
⎢   4⋅ξ - 1            0       ⎥
⎢                              ⎥
⎢      0            4⋅η - 1    ⎥
⎢                              ⎥
⎢-4⋅η - 8⋅ξ + 4       -4⋅ξ     ⎥
⎢                              ⎥
⎢     4⋅η             4⋅ξ      ⎥
⎢                              ⎥
⎣     -4⋅η       -8⋅η - 4⋅ξ + 4⎦


In [24]:
# put the triangle in the global matrix
def put_local_to_global(global_matrix, local_matrix, nodes):

       '''
       This method puts local matrices into the corresponding global ones according to their position in the domain

       :param global_matrix: the global matrix (the matrix of the domain) either empty if the first node or already have previous local matrices in it
       :param local_matrix: the matrix of the triangle
       :param coord: the coordinates of the triangle in the domain

       :return: return the obtained global matrix
       '''

       n_local = local_matrix.shape[0]

       # we need to put every value of the local matrix to the global, that's why we need 2 loops: one for rows, another for columns
       for a in range(n_local):
              for b in range(n_local):
                     global_matrix[nodes[a], nodes[b]] += local_matrix[a, b]
       

       return global_matrix



In [25]:

def compute_global_matrices(p2_triangles_coords, p2_triangles_nodes, 
                            quadrature_coords, weights, grad_func, phi_func):
       

       n_nodes = int(p2_triangles_nodes.max()) + 1  
       A_global = np.zeros((n_nodes, n_nodes), dtype=float)
       M_global = np.zeros((n_nodes, n_nodes), dtype=float) 
          
       # for every triangle
       for triangle, nodes in zip(p2_triangles_coords, p2_triangles_nodes):
              x1, y1 = triangle[0] # vertex 1
              x2, y2 = triangle[1] # vertex 2
              x3, y3 = triangle[2] # vertex 3

              # Jacobian matrix
              J = np.array(
                     [
                            [x2 - x1, x3 - x1],
                            [y2 - y1, y3 - y1]
                     ]
              )
              det_J = abs(np.linalg.det(J))

              # Metric tensor
              G = np.dot(np.linalg.inv(J), np.linalg.inv(J).T)
              
              # local matrices
              A_e, M_e = get_local_A_and_M_matrices(grad_func, phi_func, quadrature_coords, weights, det_J, G)

              # assembly
              A_global = put_local_to_global(A_global, A_e, nodes)
              M_global = put_local_to_global(M_global, M_e, nodes)

       # checks
       assert np.allclose(A_global.sum(axis=1), 0)
       assert np.allclose(A_global, A_global.T)
       assert np.allclose(M_global, M_global.T)
       
       return A_global, M_global
              


def get_local_A_and_M_matrices(grad_func, phi_func, quadrature_coords,
                            weights, det_J, G):
       A_e = np.zeros((6, 6))
       M_e = np.zeros((6, 6))

       for i, (xi_q, eta_q) in enumerate(quadrature_coords):
              
              # evaluate gradients at the reference coordinates and put them into numpy matrix
              B = np.array(grad_func(xi_q, eta_q), dtype=float)
              # check if the column sum is zero
              assert np.allclose(B.sum(axis=0), 0)

              # get local stiffness matrix
              A_e += (weights[i] / 2) * det_J * np.dot(np.dot(B, G), B.T)

              # evaluate basis functions at the reference coordinates and put them into numpy matrix
              phi_q = np.array(phi_func(xi_q, eta_q), dtype=float)
              # get local mass matrix
              M_e += (weights[i] / 2) * det_J * np.outer(phi_q, phi_q)

       # check if the matrix is symmetric
       assert np.allclose(A_e, A_e.T)
       assert np.allclose(A_e.sum(axis=1), 0)

       assert np.allclose(M_e, M_e.T)

       return A_e, M_e
       

In [26]:
def gaussian_quadrature_constants():
       # verified degree-4, 6-point Hammer–Marlowe–Stroud rule
       a1, b1, g1 = 0.816847572980459, 0.091576213509771, 0.109951743655322
       a2, b2, g2 = 0.108103018168070, 0.445948490915965, 0.223381589678011

       quadrature_coords = np.array(
              [
                     [b1, b1],
                     [a1, b1],
                     [b1, a1],
                     [b2, b2],
                     [a2, b2], 
                     [b2, a2]
              ]
       )
       # weights for the 6 point quadrature
       weights = np.array([g1, g1, g1, g2, g2, g2])

       return quadrature_coords, weights

In [27]:

n_points = 3
# generate mesh and split it into triangles
domain, triangles = generate_mesh(n_points)
# sort nodes of each triangle into ascending order so they are consistently ordered
triangles_nodes = np.sort(triangles.simplices)

mid_pts = add_mid_pts(domain, triangles_nodes)

p2_triangles_nodes, p2_triangles_coords = get_p2_triangles(domain, triangles_nodes, mid_pts)


# we take quadrature of 6 points
quadrature_coords, weights = gaussian_quadrature_constants()

# derive basis functions
# a 6x1 matrix of basis functions on the reference triangle
B_phi = derive_p2_basis_func()
B_grad_phi = basis_func_grad(B_phi)

# will evaluate the sumpy matrix, like it evaluates the numpy matrix
grad_func = sp.lambdify(
       (xi, eta),
       B_grad_phi,
       "numpy"
)

phi_func = sp.lambdify(
       (xi, eta),
       B_phi,
       "numpy"
)

A_global, M_global = compute_global_matrices(p2_triangles_coords, p2_triangles_nodes, quadrature_coords, weights, grad_func, phi_func)

# will get 25x25 matrices
print(A_global)
print(M_global)

              




[[ 1.00000000e+00  1.66666667e-01  0.00000000e+00  1.66666667e-01
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  1.88737914e-15  0.00000000e+00  0.00000000e+00
  -6.66666667e-01 -6.66666667e-01  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 1.66666667e-01  2.00000000e+00  1.66666667e-01  0.00000000e+00
   3.33333333e-01  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00 -1.04777298e-15  9.43689571e-16 -1.33333333e+00
  -6.66666667e-01  1.04083409e-16  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  9.36750677e-16 -1.87350135e-15
  -6.66666667e-01  9.36750677e-16  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 0.00000000e+00  1.66666667e-01  1.00000000e+00  0.00000000e+00
   0.00000000e+00  1.66666667e-01  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.0